# TPC4 - Word2Vec com Harry Potter

Este notebook treina um modelo Word2Vec sobre os textos de Harry Potter e demonstra três situações:
1. **similarity** – palavras com alta similaridade semântica
2. **no similarity** – palavras com baixa similaridade semântica
3. **doesnt match** – encontrar a palavra que não pertence ao grupo

## 1. Preparação do Corpus e Treino do Modelo

In [14]:
from gensim.models import Word2Vec

# Ler os livros
f1 = open("Harry_Potter_Camara_Secreta-br.txt", encoding="utf-8")
f2 = open("Harry Potter e A Pedra Filosofal.txt", encoding="utf-8")

sentences = []

for line in f1:
    words = line.split()
    if words:
        sentences.append(words)

for line in f2:
    words = line.split()
    if words:
        sentences.append(words)

f1.close()
f2.close()

### 1.1. Treinar Modelo Word2Vec

In [15]:
model = Word2Vec(
    sentences=sentences,
    vector_size=100,  # dimensão dos vetores
    window=5,         # tamanho da janela de contexto
    min_count=2,      # ignora palavras com frequência < 2
    sg=0,             # CBOW (0) ou Skip-gram (1)
    epochs=10,
    workers=3
)

print(f"Vocabulário do modelo: {len(model.wv)} palavras")
print(f"Vetor de 'Harry': {model.wv['Harry']}")

Vocabulário do modelo: 9525 palavras
Vetor de 'Harry': [-2.91185677e-01  5.43538392e-01  1.26364231e-01  6.32254630e-02
  1.27338467e-03 -1.39559761e-01  4.73277390e-01  8.46338868e-01
 -7.14083731e-01  4.69928831e-02 -2.65301913e-01 -6.37602687e-01
  1.92733303e-01  2.81828605e-02  5.01038432e-01 -7.01427937e-01
  6.54721856e-01 -5.62140524e-01 -4.27886605e-01 -9.99779344e-01
  2.39866152e-01  2.74528325e-01  1.31166041e+00  3.03964108e-01
 -4.15304363e-01  1.47047788e-01 -7.55572438e-01 -3.41611236e-01
 -3.38493705e-01  5.64118743e-01  1.26829529e+00 -6.02290928e-01
  4.66527253e-01 -1.10423124e+00 -2.47592926e-01  8.27666521e-01
 -6.32044971e-02  1.39471635e-01 -6.38466895e-01 -4.09739226e-01
  5.45084834e-01 -2.25412250e-01 -4.40183245e-02  6.42375350e-01
  4.51098591e-01  2.64661312e-01 -2.34431043e-01 -6.03155494e-01
  7.61659563e-01 -7.21815407e-01  4.20591645e-02 -9.54999402e-02
  1.29789829e-01 -6.02884173e-01 -2.31072709e-01 -7.53120799e-03
 -4.91841435e-02  4.11566377e-01 -4

---
## 2. Similarity – Palavras Semanticamente Próximas

Palavras que aparecem em contextos semelhantes terão vetores próximos no espaço vetorial.

In [16]:
print("=== Palavras mais similares a 'Harry' ===")
for word, score in model.wv.most_similar('Harry', topn=10):
    print(f"  {word:<20} similaridade: {score:.4f}")

=== Palavras mais similares a 'Harry' ===
  Hermione             similaridade: 0.9282
  Hagrid               similaridade: 0.9135
  cara                 similaridade: 0.9112
  surpreendeu          similaridade: 0.9058
  Mione                similaridade: 0.9025
  curvou               similaridade: 0.9017
  Fred                 similaridade: 0.9009
  desvencilhar         similaridade: 0.9002
  reuniram             similaridade: 0.9002
  aceno                similaridade: 0.8987


In [17]:
print("=== Palavras mais similares a 'Dumbledore' ===")
for word, score in model.wv.most_similar('Dumbledore', topn=10):
    print(f"  {word:<20} similaridade: {score:.4f}")

=== Palavras mais similares a 'Dumbledore' ===
  bem                  similaridade: 0.9863
  –,                   similaridade: 0.9853
  falar                similaridade: 0.9821
  você,                similaridade: 0.9804
  Dumbledore.          similaridade: 0.9789
  Mione,               similaridade: 0.9784
  Firenze              similaridade: 0.9784
  Rúbeo                similaridade: 0.9782
  mim?                 similaridade: 0.9775
  rouco                similaridade: 0.9772


In [18]:
pares_similares = [
    ('Harry', 'Rony'),
    ('Harry', 'Mione'),
    ('Hogwarts', 'escola'),
    ('bruxo', 'magia'),
    ('Voldemort', 'maligno'),
]

print("=== Similaridade entre pares ===")
for w1, w2 in pares_similares:
    try:
        sim = model.wv.similarity(w1, w2)
        print(f"  similarity('{w1}', '{w2}') = {sim:.4f}")
    except KeyError as e:
        print(f"  Palavra não encontrada no vocabulário: {e}")

=== Similaridade entre pares ===
  similarity('Harry', 'Rony') = 0.8923
  similarity('Harry', 'Mione') = 0.9025
  similarity('Hogwarts', 'escola') = 0.9341
  similarity('bruxo', 'magia') = 0.9431
  similarity('Voldemort', 'maligno') = 0.8917


In [19]:
print("=== Analogia: Harry - Gryffindor + Sonserina ===")
try:
    result = model.wv.most_similar(positive=['Harry', 'Sonserina'], negative=['Gryffindor'], topn=5)
    for word, score in result:
        print(f"  {word:<20} {score:.4f}")
except KeyError as e:
    print(f"  Palavra não encontrada: {e}")

=== Analogia: Harry - Gryffindor + Sonserina ===
  Palavra não encontrada: "Key 'Gryffindor' not present in vocabulary"


---
## 3. No Similarity – Palavras Semanticamente Distantes

Palavras que aparecem em contextos muito diferentes terão vetores distantes.

In [ ]:
pares_nao_similares = [
    ('Harry', 'frigideira'),
    ('Dumbledore', 'bacon'),
    ('bruxo', 'cozinha'),
    ('Hogwarts', 'tio'),
    ('varinha', 'cama'),
]

print("=== Pares com baixa similaridade ===")
for w1, w2 in pares_nao_similares:
    try:
        sim = model.wv.similarity(w1, w2)
        print(f"  similarity('{w1}', '{w2}') = {sim:.4f}")
    except KeyError as e:
        print(f"  Palavra não encontrada no vocabulário: {e}")

=== Pares com baixa similaridade ===
  similarity('Harry', 'frigideira') = 0.6807
  similarity('Dumbledore', 'bacon') = 0.8293
  similarity('bruxo', 'cozinha') = 0.8172
  similarity('Hogwarts', 'tio') = 0.8715
  similarity('varinha', 'cama') = 0.9493


In [ ]:
def classificar_similaridade(model, w1, w2, threshold_alto=0.5, threshold_baixo=0.1):
    try:
        sim = model.wv.similarity(w1, w2)
        if sim >= threshold_alto:
            categoria = "ALTA similaridade"
        elif sim >= threshold_baixo:
            categoria = "MEDIA similaridade"
        else:
            categoria = "BAIXA similaridade (no similarity)"
        return sim, categoria
    except KeyError as e:
        return None, f"doesnt match - {e} não está no vocabulário"

todos_os_pares = pares_similares + pares_nao_similares

print("=== Classificação de todos os pares ===")
print(f"{'Par':<35} {'Score':>8}  Categoria")
print("-" * 70)
for w1, w2 in todos_os_pares:
    sim, cat = classificar_similaridade(model, w1, w2)
    score_str = f"{sim:.4f}" if sim is not None else "  N/A "
    print(f"  ('{w1}', '{w2}')<{35 - len(w1) - len(w2) - 6}  {score_str:>8}  {cat}")

=== Classificação de todos os pares ===
Par                                    Score  Categoria
----------------------------------------------------------------------
  ('Harry', 'Rony')<20    0.8923  ALTA similaridade
  ('Harry', 'Mione')<19    0.9025  ALTA similaridade
  ('Hogwarts', 'escola')<15    0.9341  ALTA similaridade
  ('bruxo', 'magia')<19    0.9431  ALTA similaridade
  ('Voldemort', 'maligno')<13    0.8917  ALTA similaridade
  ('Harry', 'frigideira')<14    0.6807  ALTA similaridade
  ('Dumbledore', 'bacon')<14    0.8293  ALTA similaridade
  ('bruxo', 'cozinha')<17    0.8172  ALTA similaridade
  ('Hogwarts', 'tio')<18    0.8715  ALTA similaridade
  ('varinha', 'cama')<18    0.9493  ALTA similaridade


---
## 4. Doesnt Match – Palavra que Não Pertence ao Grupo

O método `doesnt_match` identifica qual palavra de uma lista não encaixa no contexto das outras.

In [ ]:
grupos = [
    ['Harry', 'Rony', 'Mione', 'Duda'],      # Duda não é de Hogwarts
    ['Gryffindor', 'Sonserina', 'Lufa-Lufa', 'Dumbledore'],  # Dumbledore não é uma casa
    ['varinha', 'vassoura', 'caldeirão', 'frigideira'],       # frigideira não é item mágico
    ['Harry', 'Hermione', 'Rony', 'Válter'],  # Válter é muggle
]

print("=== doesnt_match: encontrar a palavra intrusa ===")
for grupo in grupos:
    # Filtra apenas palavras que estão no vocabulário
    palavras_validas = [w for w in grupo if w in model.wv]
    palavras_invalidas = [w for w in grupo if w not in model.wv]
    
    if palavras_invalidas:
        print(f"  Grupo: {grupo}")
        print(f"    ⚠ Palavras fora do vocabulário: {palavras_invalidas}")
    
    if len(palavras_validas) >= 2:
        intrusa = model.wv.doesnt_match(palavras_validas)
        print(f"  Grupo: {palavras_validas}")
        print(f"    → Palavra intrusa: '{intrusa}'")
    print()

=== doesnt_match: encontrar a palavra intrusa ===
  Grupo: ['Harry', 'Rony', 'Mione', 'Duda']
    → Palavra intrusa: 'Duda'

  Grupo: ['Gryffindor', 'Sonserina', 'Lufa-Lufa', 'Dumbledore']
    ⚠ Palavras fora do vocabulário: ['Gryffindor']
  Grupo: ['Sonserina', 'Lufa-Lufa', 'Dumbledore']
    → Palavra intrusa: 'Dumbledore'

  Grupo: ['varinha', 'vassoura', 'caldeirão', 'frigideira']
    → Palavra intrusa: 'frigideira'

  Grupo: ['Harry', 'Hermione', 'Rony', 'Válter']
    → Palavra intrusa: 'Válter'



---
## 5. Tratamento de Palavras Fora do Vocabulário (doesnt match / OOV)

O que acontece quando tentamos aceder a uma palavra que não existe no vocabulário treinado?

In [ ]:
def safe_similarity(model, w1, w2):
    """
    Calcula a similaridade entre duas palavras de forma segura.
    Retorna None e uma mensagem de erro se alguma palavra não estiver no vocabulário.
    """
    fora_vocab = []
    if w1 not in model.wv:
        fora_vocab.append(w1)
    if w2 not in model.wv:
        fora_vocab.append(w2)
    
    if fora_vocab:
        return None, f"doesnt match: {fora_vocab} não está(ão) no vocabulário"
    
    sim = model.wv.similarity(w1, w2)
    return sim, "OK"


pares_teste = [
    ('Harry', 'Rony'),          # ambas no vocabulário
    ('Harry', 'Karate'),        # 'Karate' provavelmente fora
    ('Quidditch', 'vassoura'),  # tema HP
    ('smartphone', 'internet'), # palavras modernas, fora do HP
    ('Voldemort', 'maldade'),   # relação semântica
]

print("=== Teste com palavras potencialmente fora do vocabulário ===")
for w1, w2 in pares_teste:
    sim, status = safe_similarity(model, w1, w2)
    if sim is not None:
        print(f"  ('{w1}', '{w2}') → {sim:.4f} [{status}]")
    else:
        print(f"  ('{w1}', '{w2}') → {status}")

=== Teste com palavras potencialmente fora do vocabulário ===
  ('Harry', 'Rony') → 0.8923 [OK]
  ('Harry', 'Karate') → doesnt match: ['Karate'] não está(ão) no vocabulário
  ('Quidditch', 'vassoura') → doesnt match: ['Quidditch'] não está(ão) no vocabulário
  ('smartphone', 'internet') → doesnt match: ['smartphone', 'internet'] não está(ão) no vocabulário
  ('Voldemort', 'maldade') → doesnt match: ['maldade'] não está(ão) no vocabulário


In [ ]:
palavras_para_verificar = [
    'Harry', 'Hermione', 'Voldemort', 'Quidditch', 'Hogwarts',
    'smartphone', 'internet', 'pizza', 'bruxo', 'magia'
]

print("=== Verificação de palavras no vocabulário ===")
for palavra in palavras_para_verificar:
    esta = palavra in model.wv
    status = "✓ no vocabulário" if esta else "✗ doesnt match (fora do vocabulário)"
    print(f"  '{palavra}': {status}")

=== Verificação de palavras no vocabulário ===
  'Harry': ✓ no vocabulário
  'Hermione': ✓ no vocabulário
  'Voldemort': ✓ no vocabulário
  'Quidditch': ✗ doesnt match (fora do vocabulário)
  'Hogwarts': ✓ no vocabulário
  'smartphone': ✗ doesnt match (fora do vocabulário)
  'internet': ✗ doesnt match (fora do vocabulário)
  'pizza': ✗ doesnt match (fora do vocabulário)
  'bruxo': ✓ no vocabulário
  'magia': ✓ no vocabulário


---
## Resumo

| Situação | O que acontece | Exemplo |
|---|---|---|
| **similarity** | Score alto (próximo de 1) | `('Harry', 'Rony')` |
| **no similarity** | Score baixo (próximo de 0) | `('Harry', 'frigideira')` |
| **doesnt match** | Palavra fora do vocabulário ou intrusa no grupo | `'smartphone'` não está no vocab; `doesnt_match(['Harry','Rony','Mione','Duda'])` → `'Duda'` |

In [26]:
model.wv.save_word2vec_format('model_harry.txt', binary=False)
#https://projector.tensorflow.org/

In [29]:
!python3 -m gensim.scripts.word2vec2tensor -i model_harry.txt -o model_harry

2026-04-07 15:39:34,100 - word2vec2tensor - INFO - running /home/joaomoura03/venv-spln/lib/python3.12/site-packages/gensim/scripts/word2vec2tensor.py -i model_harry.txt -o model_harry
2026-04-07 15:39:34,100 - keyedvectors - INFO - loading projection weights from model_harry.txt
2026-04-07 15:39:34,772 - utils - INFO - KeyedVectors lifecycle event {'msg': 'loaded (9525, 100) matrix of type float32 from model_harry.txt', 'binary': False, 'encoding': 'utf8', 'datetime': '2026-04-07T15:39:34.771822', 'gensim': '4.4.0', 'python': '3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]', 'platform': 'Linux-6.8.0-106-generic-x86_64-with-glibc2.39', 'event': 'load_word2vec_format'}
2026-04-07 15:39:35,355 - word2vec2tensor - INFO - 2D tensor file saved to model_harry_tensor.tsv
2026-04-07 15:39:35,355 - word2vec2tensor - INFO - Tensor metadata file saved to model_harry_metadata.tsv
2026-04-07 15:39:35,355 - word2vec2tensor - INFO - finished running word2vec2tensor.py
